# The Schelling Segregation Model

> **How to run:** from the repo root, `uv run jupyter lab` then open this file.

A hands-on introduction to agent-based modelling: how mild individual preferences produce strong collective segregation.

## 1. Introduction

In 1971, the economist Thomas Schelling described a striking result: even if every individual is willing to live in a mixed neighbourhood — they just prefer *not* to be a tiny minority — the collective outcome is near-total segregation. No central planner, no overt discrimination; the pattern emerges purely from local movement decisions.

The model is simple. Agents of two groups (A and B) are placed on a grid. At each time step, every agent checks what fraction of its eight immediate neighbours share its type. If that fraction is below a **tolerance threshold τ**, the agent is *unsatisfied* and moves to a random empty cell. Satisfied agents stay put. After enough steps, clusters form and segregation stabilises.

This notebook walks through the model step by step: we will build intuition for the rules, watch segregation emerge in real time, and explore how τ controls the final outcome.

## 2. Setup

We load the default configuration and set up a reproducible random key.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import jax
import jax.numpy as jnp

from abm_geometry.config import load_config
from abm_geometry.rng import make_key
from abm_geometry.schelling import init_world, one_step, simulate
from abm_geometry.statistics.segregation import dissimilarity_index
from abm_geometry.viz.grids import plot_grid

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

cfg = load_config("../experiments/configs/default.yaml")
key = make_key(cfg.seed)
k_init, k_sim = jax.random.split(key)
print(f"Grid: {cfg.H}\u00d7{cfg.W}  |  Steps T: {cfg.T}  |  Tolerance \u03c4: {cfg.tau}  |  Density: {cfg.density}")

## 3. The Grid

The world is a **30×30 grid** where each cell is one of:
- **Empty** (white) — no agent present
- **Group A** (red) — an agent of type A
- **Group B** (blue) — an agent of type B

Initially, agents are placed at random. With `density = 0.8`, 80% of cells are occupied. Half are group A, half group B. There is no structure yet — this is the baseline before any movement.

In [ ]:
state = init_world(k_init, cfg)

fig, ax = plt.subplots(figsize=(5, 5))
plot_grid(state.soft_occupancy, title=f"Initial random placement  (D = {float(dissimilarity_index(state.soft_occupancy)):.3f})", ax=ax)
legend = [
    mpatches.Patch(color="#CC3333", label="Group A"),
    mpatches.Patch(color="#3333CC", label="Group B"),
    mpatches.Patch(facecolor="#F0F0F0", edgecolor="lightgrey", label="Empty"),
]
ax.legend(handles=legend, loc="lower right", fontsize=9, framealpha=0.85)
plt.tight_layout()
plt.show()

## 4. One Step: Satisfaction and Movement

At each time step the model does the following for every agent:

1. Count the agent's **8 Moore-neighbourhood** cells (the cells directly and diagonally adjacent).
2. Compute the fraction of those occupied cells that share the agent's type.
3. If that fraction is **≥ τ**, the agent is *satisfied* — it stays.
4. If it is **< τ**, the agent is *unsatisfied* — it relocates to a randomly chosen empty cell.

Below we apply a single step and highlight the cells that changed (red border).

In [ ]:
k1, _ = jax.random.split(key)
after_one = one_step(state, k1, cfg)

changed = np.array(jnp.abs(after_one.soft_occupancy - state.soft_occupancy).sum(axis=-1) > 0.1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_grid(state.soft_occupancy, title="Before (step 0)", ax=axes[0])
plot_grid(after_one.soft_occupancy, title="After (step 1)", ax=axes[1])

for r, c in zip(*np.where(changed)):
    axes[1].add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                     fill=False, edgecolor="crimson", linewidth=1.2))
plt.tight_layout()
plt.show()
print(f"Cells that moved: {int(changed.sum())}")

## 5. Full Simulation: Segregation Emerges

Let's run all T = 50 steps and compare the initial and final states.

We measure segregation with the **dissimilarity index** D:
$$D = \frac{1}{2} \sum_i \left|\frac{a_i}{A} - \frac{b_i}{B}\right|$$
where $a_i$ and $b_i$ are the type-A and type-B mass in cell $i$, and $A$, $B$ are the totals. D = 0 means perfect integration; D = 1 means complete segregation.

In [ ]:
_sim = jax.jit(simulate, static_argnums=(2,))
final_state = _sim(k_sim, state, cfg)

d_init  = float(dissimilarity_index(state.soft_occupancy))
d_final = float(dissimilarity_index(final_state.soft_occupancy))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_grid(state.soft_occupancy,       title=f"Initial   D = {d_init:.3f}",         ax=axes[0])
plot_grid(final_state.soft_occupancy, title=f"After {cfg.T} steps   D = {d_final:.3f}", ax=axes[1])
plt.tight_layout()
plt.show()
print(f"Dissimilarity: {d_init:.3f} \u2192 {d_final:.3f}  (+{d_final - d_init:.3f})")

## 6. Segregation Over Time

How quickly does D rise, and when does it stabilise? We track the dissimilarity index at every step.
*(The D values here may differ slightly from Section 5 because both sections use independent random streams — this is intentional and shows the model is stable across different noise realisations.)*

In [ ]:
d_curve = [float(dissimilarity_index(state.soft_occupancy))]
s = state
step_keys = jax.random.split(key, cfg.T)
for k in step_keys:
    s = one_step(s, k, cfg)
    d_curve.append(float(dissimilarity_index(s.soft_occupancy)))

# First t >= 10 where |D(t) - D(t-5)| < 0.01; fallback to step 35
stabilise_t = 35
for t in range(10, cfg.T):
    if abs(d_curve[t] - d_curve[t - 5]) < 0.01:
        stabilise_t = t
        break

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(cfg.T + 1), d_curve, color="#2b6cb0", linewidth=2)
ax.axvline(1, color="grey", linestyle="--", alpha=0.6)
ax.text(1.5, d_curve[1] - 0.04, "agents start\nmoving", fontsize=9, color="grey", va="top")
ax.axvline(stabilise_t, color="grey", linestyle="--", alpha=0.6)
ax.text(stabilise_t + 0.5, d_curve[stabilise_t] + 0.01,
        "segregation\nstabilises", fontsize=9, color="grey")
ax.set_xlabel("Step")
ax.set_ylabel("Dissimilarity index $D$")
ax.set_ylim(0, 1)
ax.set_title("Segregation grows rapidly then stabilises")
plt.tight_layout()
plt.show()

## 7. The Role of Tolerance

The key parameter is τ. A **higher** τ means agents are more demanding: they need a larger fraction of like-minded neighbours to be satisfied. Below we run the same simulation from the same starting state with τ ∈ {0.2, 0.4, 0.6}.

Notice how even a moderate tolerance (τ = 0.4 means you stay only if at least 40% of neighbours share your type) produces clear segregation — and higher τ amplifies it further. This is the core Schelling result.

In [ ]:
# requires _sim from Section 5 — run that cell first
taus = [0.2, 0.4, 0.6]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, tau_val in zip(axes, taus):
    s_tau = state.replace(tolerances=jnp.full((cfg.H, cfg.W), tau_val))
    final_tau = _sim(key, s_tau, cfg)
    d_tau = float(dissimilarity_index(final_tau.soft_occupancy))
    plot_grid(final_tau.soft_occupancy, title=f"\u03c4 = {tau_val}\nD = {d_tau:.3f}", ax=ax)

plt.suptitle(f"Final state after {cfg.T} steps — higher \u03c4 means more segregation",
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 8. Summary Statistics Vector

To study the Fisher Information Matrix we need a **vector** of summary statistics $\mathbf{s}(\theta) \in \mathbb{R}^4$, not just a scalar. We use four differentiable statistics:

| # | Stat | What it measures |
|---|------|-----------------|
| 0 | Dissimilarity $D$ | global spatial segregation level |
| 1 | Mean satisfaction | average agent happiness (fraction of similar neighbours ≥ τ) |
| 2 | Moran's $I$ | spatial autocorrelation of type-A fraction; positive = clustered |
| 3 | Neighbourhood homogeneity | fraction of neighbours sharing the cell's type — a soft cluster-size proxy |

All four are fully differentiable via the underlying convolution operations.

In [ ]:
import numpy as np
from abm_geometry.statistics.summary import stats_array_fn

STAT_NAMES = ["Dissimilarity $D$", "Mean satisfaction", "Moran's $I$", "Neighbourhood\nhomogeneity"]
COLORS = ["#2b6cb0", "#c05621", "#276749", "#6b46c1"]

# Collect stats at every step (reuse state / keys from Section 2)
state8 = init_world(k_init, cfg)
step_keys8 = jax.random.split(k_sim, cfg.T)
stats_over_time = []
s8 = state8
for k8 in step_keys8:
    s8 = one_step(s8, k8, cfg)
    stats_over_time.append(np.array(stats_array_fn(s8, cfg.beta)))
stats_arr = np.array(stats_over_time)  # (T, 4)

fig, ax = plt.subplots(figsize=(9, 4))
for i, (name, color) in enumerate(zip(STAT_NAMES, COLORS)):
    ax.plot(range(1, cfg.T + 1), stats_arr[:, i], label=name, color=color, lw=2)
ax.set_xlabel("Step")
ax.set_ylabel("Statistic value")
ax.legend(ncol=2, fontsize=9)
ax.set_title("All four summary statistics over time")
plt.tight_layout()
plt.show()

# Summary table
init_stats = np.array(stats_array_fn(state8, cfg.beta))
print(f"{'Statistic':<30} {'t=0':>8} {'t=25':>8} {'t=50':>8}")
print("-" * 58)
for i, name in enumerate([s.replace("\n", " ") for s in STAT_NAMES]):
    print(f"{name:<30} {init_stats[i]:>8.3f} {stats_arr[24, i]:>8.3f} {stats_arr[-1, i]:>8.3f}")

## 9. Fisher Information Matrix (Phase I)

The **Fisher Information Matrix** quantifies how much information the summary statistics carry about the parameters $\theta$. It is:

$$F(\theta) = J^\top \Sigma^{-1} J, \quad J = \frac{\partial \mathbf{s}}{\partial \theta} \in \mathbb{R}^{4 \times p}$$

where $\Sigma$ is the noise covariance (estimated by re-running the simulation with $K=20$ different seeds at the same $\theta$).

**Eigenvalues of $F$** tell us which parameter directions are *stiff* (large $\lambda$ — well identified) vs *sloppy* (small $\lambda$ — hard to identify). A large condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ signals a **sloppy model**.

For Phase I we use $\theta = (\tau, \beta)$ — both the tolerance threshold and the sigmoid sharpness.

In [ ]:
from abm_geometry.geometry.fim import estimate_noise_cov, fisher_information
from abm_geometry.geometry.jacobian import compute_jacobian
from abm_geometry.geometry.spectrum import condition_number, eigendecomp
from abm_geometry.schelling.simulate import simulate_with_beta

_SIM_BETA = jax.jit(simulate_with_beta, static_argnums=(2,))

key9 = jax.random.PRNGKey(77)
k9_init, k9_sim, k9_cov = jax.random.split(key9, 3)
params9 = jnp.array([0.4, 5.0])   # [tau, beta]

def stats_from_params9(params):
    tau, beta = params[0], params[1]
    state = init_world(k9_init, cfg).replace(tolerances=jnp.full((cfg.H, cfg.W), tau))
    final = _SIM_BETA(k9_sim, state, cfg, beta)
    return stats_array_fn(final, beta)

# Extract concrete Python floats before the closure so vmap doesn't trace them
_tau9, _beta9 = float(params9[0]), float(params9[1])

def run_fn9(key):
    ki, ks = jax.random.split(key)
    state = init_world(ki, cfg).replace(tolerances=jnp.full((cfg.H, cfg.W), _tau9))
    final = _SIM_BETA(ks, state, cfg, _beta9)
    return stats_array_fn(final, _beta9)

print("Computing Jacobian…")
J9 = compute_jacobian(stats_from_params9, params9)          # Float[4, 2]
print("Estimating noise covariance (K=20 runs)…")
Sigma9 = estimate_noise_cov(run_fn9, k9_cov, K=20)
F9 = fisher_information(J9, Sigma9)                          # Float[2, 2]
vals9, vecs9 = eigendecomp(F9)
kappa9 = condition_number(vals9)

stat_labels = [s.replace("\n", " ") for s in STAT_NAMES]
print(f"\nJacobian J (4 stats × 2 params [τ, β]):")
for i, name in enumerate(stat_labels):
    print(f"  {name:<32}  ∂/∂τ={float(J9[i,0]):+.4f}   ∂/∂β={float(J9[i,1]):+.4f}")

print(f"\nFIM  F = J^T Σ^{{-1}} J:")
print(np.array(F9).round(3))
print(f"\nEigenvalues:  λ₁={float(vals9[0]):.4f},  λ₂={float(vals9[1]):.6f}")
print(f"Condition number  κ = {float(kappa9):.1f}")
if float(kappa9) > 100:
    print("→ Sloppy model: parameter directions are poorly co-identified.")
else:
    print("→ Well-conditioned: both parameters identifiable at this point.")

## 10. FIM Landscape over (τ, β)

We sweep $\tau \in [0.1, 0.8]$ and $\beta \in [2, 15]$ on a $15 \times 15$ grid, computing the full FIM at each point.

- **Left:** condition number $\kappa(\tau, \beta)$ on a log scale. Bright = sloppy; dark = stiff.
- **Right:** eigenvector quivers. Blue = stiff direction (most information); red = sloppy direction (least information).

*Takes ~3–5 minutes — the simulation is jit-compiled so each point runs in seconds.*

In [ ]:
from abm_geometry.viz.landscape import plot_condition_heatmap, plot_eigenvector_quivers

tau_vals  = np.linspace(0.1, 0.8, 15)
beta_vals = np.linspace(2.0, 15.0, 15)

key10 = jax.random.PRNGKey(101)
k10_init, k10_sim, k10_cov = jax.random.split(key10, 3)

kappa_grid10 = np.zeros((15, 15))
stiff_vecs10  = np.zeros((15, 15, 2))
sloppy_vecs10 = np.zeros((15, 15, 2))

for i, tau_v in enumerate(tau_vals):
    for j, beta_v in enumerate(beta_vals):
        params_ij = jnp.array([tau_v, beta_v])

        def _pipe_ij(p, ki=k10_init, ks=k10_sim, c=cfg):
            t, b = p[0], p[1]
            s = init_world(ki, c).replace(tolerances=jnp.full((c.H, c.W), t))
            return stats_array_fn(_SIM_BETA(ks, s, c, b), b)

        def _run_ij(key, tv=tau_v, bv=beta_v, c=cfg):
            ki, ks = jax.random.split(key)
            s = init_world(ki, c).replace(tolerances=jnp.full((c.H, c.W), tv))
            return stats_array_fn(_SIM_BETA(ks, s, c, bv), bv)

        J_ij   = compute_jacobian(_pipe_ij, params_ij)
        Sig_ij = estimate_noise_cov(_run_ij, k10_cov, K=20)
        F_ij   = fisher_information(J_ij, Sig_ij)
        v_ij, e_ij = eigendecomp(F_ij)

        kappa_grid10[i, j]  = float(condition_number(v_ij))
        stiff_vecs10[i, j]  = np.array(e_ij[:, 0])
        sloppy_vecs10[i, j] = np.array(e_ij[:, 1])
    print(f"τ={tau_v:.2f} ✓", end="  ", flush=True)

fig10, axes10 = plt.subplots(1, 2, figsize=(13, 5))
plot_condition_heatmap(tau_vals, beta_vals, kappa_grid10,
                       xlabel="τ", ylabel="β", ax=axes10[0],
                       title="log₁₀ κ(τ, β)")
plot_eigenvector_quivers(tau_vals, beta_vals, stiff_vecs10, sloppy_vecs10, ax=axes10[1])
axes10[1].set_xlabel("β"); axes10[1].set_ylabel("τ")
axes10[1].set_title("Stiff (blue) / sloppy (red) eigenvectors")
plt.suptitle("Phase I FIM landscape", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 11. Heterogeneous Agents (Phase II)

So far every agent had the same tolerance $\tau$. In reality, individuals differ. We now model tolerance as **normally distributed**: $\tau_{ij} \sim \mathcal{N}(\mu, \sigma^2)$, clipped to $[0.001, 0.999]$.

This introduces a second scientific parameter $\sigma$ — the **heterogeneity** of the population. The reparametrisation trick ($\tau_{ij} = \mu + \sigma\varepsilon_{ij}$, $\varepsilon \sim \mathcal{N}(0,1)$) keeps gradients $\partial/\partial\mu$ and $\partial/\partial\sigma$ flowing correctly.

Our Phase II parameter vector is $\theta = (\mu, \sigma, \beta) \in \mathbb{R}^3$, giving a $3 \times 3$ FIM.

In [ ]:
from abm_geometry.config import Config
from abm_geometry.schelling.state import init_world_heterogeneous
from abm_geometry.viz.grids import plot_grid_comparison

cfg_het = Config(H=cfg.H, W=cfg.W, T=cfg.T, density=cfg.density,
                 tau=0.4, sigma_tau=0.15, seed=cfg.seed)

key11 = jax.random.PRNGKey(11)
k11a, k11b = jax.random.split(key11)

state_hom = init_world(k11a, cfg)
state_het = init_world_heterogeneous(k11b, cfg_het)
final_hom = _SIM_BETA(k11a, state_hom, cfg, jnp.array(cfg.beta))
final_het = _SIM_BETA(k11b, state_het, cfg_het, jnp.array(cfg_het.beta))

plot_grid_comparison(
    [final_hom, final_het],
    [f"Homogeneous (σ=0)\nD={float(dissimilarity_index(final_hom.soft_occupancy)):.3f}",
     f"Heterogeneous (σ=0.15)\nD={float(dissimilarity_index(final_het.soft_occupancy)):.3f}"],
)
plt.suptitle("Same μ=0.4, β=5 — effect of tolerance heterogeneity", y=1.02)
plt.show()

# Tolerance distribution
fig11, ax11 = plt.subplots(figsize=(6, 3))
tol_vals = np.array(state_het.tolerances).ravel()
ax11.hist(tol_vals[tol_vals > 0.01], bins=25, color="#276749", edgecolor="white", alpha=0.85)
ax11.axvline(0.4, color="red", lw=1.5, ls="--", label="μ = 0.4")
ax11.set_xlabel("Tolerance τ"); ax11.set_ylabel("Count")
ax11.set_title("Tolerance distribution (σ=0.15)")
ax11.legend(); plt.tight_layout(); plt.show()

# D-curve comparison
d_hom = [float(dissimilarity_index(state_hom.soft_occupancy))]
d_het = [float(dissimilarity_index(state_het.soft_occupancy))]
sh, sh2 = state_hom, state_het
step_keys11 = jax.random.split(key11, cfg.T)
for kk in step_keys11:
    sh  = one_step(sh,  kk, cfg)
    sh2 = one_step(sh2, kk, cfg_het)
    d_hom.append(float(dissimilarity_index(sh.soft_occupancy)))
    d_het.append(float(dissimilarity_index(sh2.soft_occupancy)))

fig11b, ax11b = plt.subplots(figsize=(8, 4))
ax11b.plot(d_hom, label="Homogeneous (σ=0)",      color="#2b6cb0", lw=2)
ax11b.plot(d_het, label="Heterogeneous (σ=0.15)",  color="#c05621", lw=2, ls="--")
ax11b.set_xlabel("Step"); ax11b.set_ylabel("Dissimilarity $D$")
ax11b.legend(); ax11b.set_title("Effect of tolerance heterogeneity on segregation dynamics")
plt.tight_layout(); plt.show()

## 12. FIM Landscape (Phase II)

With $\theta = (\mu, \sigma, \beta)$ and $\beta$ fixed at 5, we sweep $(\mu, \sigma)$ on a $12 \times 12$ grid. The FIM is now $3 \times 3$; we project stiff/sloppy directions onto the $(\mu, \sigma)$ plane.

**Key question:** does $\sigma$ add a genuinely new sloppy direction, or are $\mu$ and $\sigma$ well-separated in information content?

In [ ]:
from abm_geometry.viz.spectra import plot_eigenvalue_spectrum

mu_vals    = np.linspace(0.1, 0.8, 12)
sigma_vals = np.linspace(0.01, 0.3, 12)
BETA_FIXED = 5.0

key12 = jax.random.PRNGKey(202)
k12_init, k12_sim, k12_cov = jax.random.split(key12, 3)

kappa_grid12 = np.zeros((12, 12))
stiff12  = np.zeros((12, 12, 2))
sloppy12 = np.zeros((12, 12, 2))

SPEC_POINTS = {
    "(μ=0.2, σ=0.05)": (1, 1),
    "(μ=0.7, σ=0.05)": (10, 1),
    "(μ=0.2, σ=0.25)": (1, 10),
    "(μ=0.7, σ=0.25)": (10, 10),
}
spec_eigs = {}

for i, mu_v in enumerate(mu_vals):
    for j, sig_v in enumerate(sigma_vals):
        params_ij = jnp.array([mu_v, sig_v, BETA_FIXED])

        def _pipe12(p, ki=k12_init, ks=k12_sim, c=cfg):
            mu, sigma, beta = p[0], p[1], p[2]
            eps = jax.random.normal(ki, (c.H, c.W))
            tol = jnp.clip(mu + sigma * eps, 0.001, 0.999)
            s = init_world(ki, c).replace(tolerances=tol)
            return stats_array_fn(_SIM_BETA(ks, s, c, beta), beta)

        def _run12(key, mv=mu_v, sv=sig_v, bv=BETA_FIXED, c=cfg):
            ki, ks = jax.random.split(key)
            eps = jax.random.normal(ki, (c.H, c.W))
            tol = jnp.clip(mv + sv * eps, 0.001, 0.999)
            s = init_world(ki, c).replace(tolerances=tol)
            return stats_array_fn(_SIM_BETA(ks, s, c, bv), bv)

        J12   = compute_jacobian(_pipe12, params_ij)
        Sig12 = estimate_noise_cov(_run12, k12_cov, K=20)
        F12   = fisher_information(J12, Sig12)
        v12, e12 = eigendecomp(F12)

        kappa_grid12[i, j] = float(condition_number(v12))
        stiff12[i, j]  = np.array(e12[:2, 0])
        sloppy12[i, j] = np.array(e12[:2, 1])

        for label, (ri, rj) in SPEC_POINTS.items():
            if i == ri and j == rj:
                spec_eigs[label] = v12
    print(f"μ={mu_v:.2f} ✓", end="  ", flush=True)

# Heatmap + quivers
fig12, axes12 = plt.subplots(1, 2, figsize=(13, 5))
plot_condition_heatmap(mu_vals, sigma_vals, kappa_grid12,
                       xlabel="μ", ylabel="σ", ax=axes12[0],
                       title="log₁₀ κ(μ, σ)  [β=5 fixed]")
plot_eigenvector_quivers(mu_vals, sigma_vals, stiff12, sloppy12, ax=axes12[1])
axes12[1].set_xlabel("σ"); axes12[1].set_ylabel("μ")
axes12[1].set_title("Stiff (blue) / sloppy (red) in (μ, σ) plane")
plt.suptitle("Phase II FIM landscape", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Eigenvalue spectrum — Brown/Sethna style
if spec_eigs:
    fig12b, ax12b = plt.subplots(figsize=(6, 4))
    plot_eigenvalue_spectrum(
        list(spec_eigs.values()),
        list(spec_eigs.keys()),
        ax=ax12b,
        title="FIM eigenvalue spectrum at 4 (μ, σ) points",
    )
    plt.tight_layout()
    plt.show()

    print("\nCondition numbers at representative points:")
    for label, v in spec_eigs.items():
        print(f"  {label}: κ = {float(condition_number(v)):.1f}")

## 13. Mean-Field vs Stochastic FIM

The Gumbel-softmax step introduces stochasticity: at each step, agents sample a random stay/move decision. In the **mean-field limit** we replace that sample with its expected value — `p_stay = σ(β·(2·sat−1)/τ_g)` — giving a fully deterministic trajectory. The resulting mean-field FIM is `F_mf = J_mf^T J_mf` (Σ=I, no Monte Carlo needed).

Comparing `F_mf` and `F_sim` answers the core question: **is sloppiness a property of the model equations, or an artefact of stochastic fluctuations?**

- If `κ_mf ≈ κ_sim` everywhere → sloppiness is deterministic; mean-field theory is sufficient.
- If `κ_sim >> κ_mf` → stochasticity amplifies sloppiness; noise matters for identifiability.
- If `κ_sim << κ_mf` → stochasticity reduces sloppiness; noise actually helps identifiability.

In [ ]:
from abm_geometry.theory.mean_field import simulate_mean_field_with_params
from abm_geometry.viz.comparison import (
    plot_kappa_comparison, plot_kappa_ratio,
)

_SIM_MF = jax.jit(simulate_mean_field_with_params, static_argnums=(1,))

# Reuse tau_vals, beta_vals, kappa_grid10 from Section 10
kappa_grid_mf = np.zeros((15, 15))

print("Computing mean-field FIM landscape (no noise cov needed)…")
for i, tau_v in enumerate(tau_vals):
    for j, beta_v in enumerate(beta_vals):
        params_ij = jnp.array([tau_v, beta_v])

        def _pipe_mf13(p, ki=k10_init, c=cfg):
            t, b = p[0], p[1]
            s = init_world(ki, c)
            return stats_array_fn(_SIM_MF(s, c, t, b), b)

        J_mf_ij = compute_jacobian(_pipe_mf13, params_ij)
        F_mf_ij = J_mf_ij.T @ J_mf_ij        # Σ = I
        v_mf_ij, _ = eigendecomp(F_mf_ij)
        kappa_grid_mf[i, j] = float(condition_number(v_mf_ij))

    print(f"τ={tau_v:.2f} ✓", end="  ", flush=True)

print(f"\nMean-field range: κ ∈ [{kappa_grid_mf.min():.1f}, {kappa_grid_mf.max():.1f}]")
print(f"Stochastic range: κ ∈ [{kappa_grid10.min():.1f}, {kappa_grid10.max():.1f}]")

In [ ]:
# ── Side-by-side condition number heatmaps ────────────────────────────────────
fig13a = plot_kappa_comparison(
    tau_vals, beta_vals, kappa_grid10, kappa_grid_mf, xlabel="τ", ylabel="β"
)
plt.suptitle("Stochastic vs mean-field condition number κ(τ, β)", fontsize=13, y=1.02)
plt.show()

# ── Ratio heatmap: where does stochasticity matter? ────────────────────────────
fig13b, ax13b = plt.subplots(figsize=(7, 4))
plot_kappa_ratio(
    tau_vals, beta_vals, kappa_grid10, kappa_grid_mf,
    xlabel="τ", ylabel="β", ax=ax13b,
)
ax13b.set_title("log₁₀(κ_sim / κ_mf) — positive = stochasticity amplifies sloppiness")
plt.tight_layout()
plt.show()

# ── Summary statistics ─────────────────────────────────────────────────────────
ratio_flat = (np.log10(np.clip(kappa_grid10, 1, None))
              - np.log10(np.clip(kappa_grid_mf, 1, None)))
max_ij = np.unravel_index(ratio_flat.argmax(), ratio_flat.shape)
print("\n=== Scientific conclusion ===")
print(f"Mean log₁₀(κ_sim/κ_mf) = {ratio_flat.mean():+.2f}")
print(f"  (+) stochasticity amplifies sloppiness on average" if ratio_flat.mean() > 0.1
      else f"  (~) mean-field closely approximates the stochastic FIM")
print(f"Max deviation: {ratio_flat.max():+.2f} at "
      f"τ={tau_vals[max_ij[0]]:.2f}, β={beta_vals[max_ij[1]]:.1f}")